# 05_03 Memorising vs understanding

*📖 Book: sections 5.1.3 (training and validation losses) and 5.2*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** The best loss of 05_02, 0.008, was measured on the same 5 lines the model read 800 times. Did it learn, or just memorise?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a low loss can mean the model only memorised the text
- What happens when the model meets text it has never seen
- How to test a model on hidden text: the validation set
- How to cut the data into a training part and a validation part

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **training loss** | the loss on the text the model learns from | 0.008 on the 5 lines, after 800 epochs |
| **overfitting** | memorising the training text instead of learning its patterns | a student who memorised one exam |
| **generalisation** | doing well on text the model has never seen | writing good Greek after a new word |
| **OOV** | out of vocabulary: a character the tokenizer has no number for | `Ὀ` for the 5-line model |
| **training set** | the part of the text the model learns from | the first 90% |

Words you've met before: **validation set** ([05_00](../05_training_on_the_odyssey/05_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ How small is the practice text?

Five lines, 234 characters. The best run of [05_02](05_02_learning_rate.ipynb) (cosine schedule, 800 epochs) ended with an average loss of 0.008. To get there, the model read those same 5 lines 800 times.

### ❓ Is a low loss on 234 characters a good thing?

No. Imagine a student who gets the same 5-question test 800 times. They'll score 100%, but they may just have memorised "answer 3 is B" without understanding anything. When a model memorises like that, it's called **overfitting**.

### ❓ What does that model do with text it has never seen?

Let's train it again and ask. We give it two starting words. A starting text like that is called a **prompt**: the text you give a model to continue. It then writes by guessing one character, adding it to the end, and guessing again (chapter 6 explains this in full; `write` with `top_k=1` always takes the most likely character).

```
the 5 lines start:   ἄνδρα μοι ἔννεπε, μοῦσα, πολύτροπον, ὃς μάλα πολλὰ
prompt 1:            ἄνδρα      <- the first word of the 5 lines
prompt 2:            Ὀδυσσεύς   <- Odysseus: not in the 5 lines at all
```

`tiny_model` is the training run of 05_02 (the same seed and settings), and it hands back the model too. It takes about 7 seconds and saves nothing. A line break in the output is shown as `⏎`.

👉 **Predict first:** what will the model write after `ἄνδρα`? And after a word it has never read?

In [ ]:
from homer_ai.lessons import tiny_model, write

# the best run of 05_02: 5 lines, cosine schedule, 800 epochs
model, five_line_tokenizer, losses = tiny_model(seed=42, epochs=800, cosine=True)
print("final training loss:", round(losses[-1], 3))
for prompt in ["ἄνδρα", "Ὀδυσσεύς"]:
    print(write(model, five_line_tokenizer, prompt, 40, top_k=1))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

```
ἄνδρα μοι ἔννεπε, μοῦσα, α, μοῦσαμοῦσαῦσαῦσαῦ
<|unk|>δυσσεύς,ολα, α, μυ, μλμλμλμλμλμοῦσα α α α α ν γ
```

- After `ἄνδρα` the model **recites** line 1 word for word (`μοι ἔννεπε, μοῦσα,`). Then it loses its place and goes round in circles (`μοῦσαμοῦσαῦσαῦσα`).
- `Ὀδυσσεύς` isn't in the 5 lines, and the model writes gibberish. Its first letter, `Ὀ`, isn't even in the vocabulary. It's **OOV**, so the tokenizer turned it into `<|unk|>` ([02_02](../02_text_to_numbers/02_02_special_tokens.ipynb)).

So the loss of 0.008 didn't mean the model knows Greek. It means it knows these 5 lines by heart.

</details>

### ❓ So how do we measure real learning?

Keep some text hidden. The model trains on one part, the **training set**, and is only tested on the other part, the validation set. Think of an exam with questions it has never seen.

```
training loss ↓   validation loss ↓   → it's learning real patterns
training loss ↓   validation loss ↑   → it's memorising (overfitting)
```

### ❓ Should we train on the whole book, not just 5 lines?

Yes. More text is the best cure for memorising. With 5 lines, memorising is all the model can do. With the whole of Book 1 (444 lines, 19,701 characters), the same patterns show up again and again in different places, so learning the pattern becomes the easiest way to lower the loss.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How small is the 5-line text, and is `Ὀ` in it?

`in` checks whether something is inside a text. 👉 **Predict first:** is `Ὀ` in the 5 lines? In Book 1?

In [ ]:
from homer_ai.train_homer import BOOKS_DIR, load_books

book1 = load_books(BOOKS_DIR)[0]
five_lines = "\n".join(book1.splitlines()[:5])

print("characters in 5 lines:", len(five_lines))
print("Ὀ in the 5 lines?", "Ὀ" in five_lines)
print("Ὀ in Book 1?", "Ὀ" in book1)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So, 234 characters, and no `Ὀ` among them. That's why the 5-line model had never heard of it: it was out of vocabulary, and the tokenizer could only give it `<|unk|>`. Give the model all of Book 1 and the problem goes away.

</details>

### ❓ How do I cut the data into two parts?

Pick a cut position `n` at the 80% mark. Everything before `n` is the first part, and everything from `n` on is the second.

👉 **Predict first:** with 10 numbers and an 80% split, what are the two parts?

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
n = int(0.8 * len(numbers))  # the 80% mark
print("n:", n)
print("first part:", numbers[:n])
print("second part:", numbers[n:])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`n` is 8, so the first part has 8 numbers and the second part has 2. Nothing is lost and nothing is repeated.

</details>

### ❓ Why take the last block as the test, and not random letters?

Because text has an order. If the test letters were picked at random, each one would sit right next to training letters, so the "unseen" text would hardly be unseen. A whole block at the end is a real passage the model has never read.

👉 **Predict first:** what's the very last line of Book 1?

In [ ]:
print(book1.splitlines()[-1])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

That's the last line of Book 1. With the split, the model never trains on it. It's only tested on it.

</details>

## 5. Full example: splitting Book 1

`load_data` gives us all of Book 1 as numbers. We keep the first 90% for training and the last 10% for validation. `.tolist()` turns a tensor into a plain Python list, so we can decode it.

```
Book 1:  ἄνδρα μοι ἔννεπε … … … … … … … … … … … … … | … ὁδὸν τὴν πέφραδʼ Ἀθήνη.
         └────────────── training (first 90%) ──────────┘ └── validation (last 10%) ──┘
```

👉 **Predict first:** Book 1 has 19,701 characters. How many go to validation?

In [ ]:
from homer_ai.train_homer import load_data

data, tokenizer, text = load_data(BOOKS_DIR)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("all data:", len(data))
print("training:", len(train_data))
print("validation:", len(val_data))
print("together:", len(train_data) + len(val_data))
print()
print("start of the validation text:")
print(tokenizer.decode(val_data[:80].tolist()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

17,730 characters for training and 1,971 for validation, which add up to all 19,701. The validation text is about the last 44 lines of Book 1.

The small model's `train()` doesn't make this split: it trains on all of Book 1. Doing the split for real, with `evaluate()` and a validation loss, is [05_05](05_05_validation.ipynb).

</details>

### ❓ Can I see the split?

One long bar is Book 1. The orange end is the part the model never trains on.

👉 **Predict first:** will the orange part look much smaller than the grey part?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, MUTED, studio_axes, studio_figure, text_on

fig = studio_figure(
    "Book 1, cut in two",
    f"{len(train_data):,} characters to learn from, {len(val_data):,} kept hidden",
    size=(9, 5),
)
ax = studio_axes(fig, (0.07, 0.2, 0.86, 0.5))
ax.barh([0], [len(train_data)], color=MUTED, height=0.6)
ax.barh([0], [len(val_data)], left=len(train_data), color=ACCENT, height=0.6)
ax.text(
    len(train_data) / 2, 0, f"training: first {len(train_data) / len(data):.0%}",
    ha="center", va="center", color=text_on(1, 1), fontsize=14, fontweight="bold",
)
ax.text(
    len(data), 0.38, f"validation: last {len(val_data) / len(data):.0%}",
    ha="right", va="bottom", color=ACCENT, fontsize=13, fontweight="bold",
)
ax.set_xlim(0, len(data))
ax.set_ylim(-0.4, 0.8)
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("position in the text (characters)")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The orange validation part is a small slice at the end, about one character in ten. It's still big enough to be a fair test, and it's text the model has never read.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Is 10% enough for the validation set?

For Book 1, 10% is 1,971 characters. That's enough to see whether the model learned, but it's small, so the number wobbles a little from run to run. With more books, the validation set grows and the number gets steadier.

### ❓ Why did the model go round in circles (`μοῦσαμοῦσαῦσαῦσα`)?

It learned "what comes next" only for paths it had seen. Once it fell off the memorised path, it went round in circles.

## 6. Recap

- A low training loss can mean memorising (overfitting), not understanding.
- Text the model has never seen shows the truth: our 5-line model recited, looped, or wrote gibberish.
- So keep a block at the end hidden (the validation set), and measure on it.

**You can now:**

- test a model on text it has never seen
- tell memorising from learning by what the model writes

**Next:** [05_04: Running fair experiments](05_04_running_fair_experiments.ipynb): how to change one thing and know what it did.